# IS3107 Chess ML Model

This notebook is organized as a simple pipeline:

1. Install dependencies
2. Configure the run
3. Define helper functions
4. Download and parse Lichess PGN data
5. Inspect missing evaluation coverage
6. Fill missing evaluations with Stockfish
7. Inspect the filled dataset before feature engineering

Notes:

- Existing Lichess `%eval` annotations are preserved.
- Stockfish is only used for rows where both `eval_cp` and `mate_in` are missing.
- Full-engine scoring on every missing row can be expensive. Start with smaller settings, validate, then scale up.
        


In [13]:
!pip install requests zstandard python-chess pandas


In [15]:
# Run configuration
FEB_2026_URL = "https://database.lichess.org/broadcast/lichess_db_broadcast_2026-02.pgn.zst"

# Parsing config
MAX_GAMES = 10000

# Engine-eval config
FILL_MISSING_EVALS = True
STOCKFISH_PATH = None  # Set this manually if stockfish is not on PATH.
ENGINE_DEPTH = 1       # Use shallow depth for scalable feature generation.
ENGINE_TIME_LIMIT = 0.01
ENGINE_MAX_ROWS = None  # Set to an int for a quick test run.
        


In [17]:
import io
import os
import re
import shutil
from typing import Dict, List, Optional

import chess
import chess.engine
import chess.pgn
import pandas as pd
import requests
import zstandard as zstd
from tqdm import tqdm


DEFAULT_STOCKFISH_CANDIDATES = [
    STOCKFISH_PATH,
    os.environ.get("STOCKFISH_PATH"),
    shutil.which("stockfish"),
    "/opt/homebrew/bin/stockfish",
    "/usr/local/bin/stockfish",
]


def stream_text_from_zst_url(url: str) -> io.TextIOBase:
    resp = requests.get(url, stream=True, timeout=120)
    resp.raise_for_status()

    dctx = zstd.ZstdDecompressor()
    reader = dctx.stream_reader(resp.raw)
    return io.TextIOWrapper(reader, encoding="utf-8", errors="replace")


def parse_clock_to_seconds(clock_str: str) -> Optional[float]:
    parts = clock_str.split(":")
    try:
        parts = [float(x) for x in parts]
    except ValueError:
        return None

    if len(parts) == 3:
        h, m, s = parts
        return h * 3600 + m * 60 + s
    if len(parts) == 2:
        m, s = parts
        return m * 60 + s
    if len(parts) == 1:
        return parts[0]
    return None


def extract_eval_and_clock(comment: str):
    eval_cp = None
    mate_in = None
    clk_seconds = None

    eval_match = re.search(r"\[%eval\s+([^\]]+)\]", comment)
    clk_match = re.search(r"\[%clk\s+([^\]]+)\]", comment)

    if eval_match:
        raw_eval = eval_match.group(1).strip()
        if raw_eval.startswith("#"):
            try:
                mate_in = int(raw_eval[1:])
            except ValueError:
                mate_in = None
        else:
            try:
                eval_cp = int(round(float(raw_eval) * 100))
            except ValueError:
                eval_cp = None

    if clk_match:
        clk_seconds = parse_clock_to_seconds(clk_match.group(1).strip())

    return eval_cp, mate_in, clk_seconds


def result_to_label(result: str) -> str:
    if result == "1-0":
        return "white_win"
    if result == "0-1":
        return "black_win"
    if result == "1/2-1/2":
        return "draw"
    return "unknown"


def find_stockfish_binary(stockfish_path: Optional[str] = None) -> str:
    candidates = [stockfish_path, *DEFAULT_STOCKFISH_CANDIDATES]
    for candidate in candidates:
        if candidate and os.path.exists(candidate):
            return candidate

    raise FileNotFoundError(
        "Stockfish binary not found. Install Stockfish locally and set STOCKFISH_PATH if needed."
    )


def score_fen_with_stockfish(
    engine: chess.engine.SimpleEngine,
    fen: str,
    depth: int = 1,
    time_limit: Optional[float] = 0.01,
):
    board = chess.Board(fen)
    limit = chess.engine.Limit(depth=depth) if time_limit is None else chess.engine.Limit(depth=depth, time=time_limit)
    info = engine.analyse(board, limit)
    score = info["score"].white()

    if score.is_mate():
        return None, score.mate()
    return score.score(mate_score=100000), None


def fill_missing_evals_with_stockfish(
    df: pd.DataFrame,
    stockfish_path: Optional[str] = None,
    depth: int = 1,
    time_limit: Optional[float] = 0.01,
    max_rows: Optional[int] = None,
) -> pd.DataFrame:
    missing_mask = df["eval_cp"].isna() & df["mate_in"].isna()
    missing_idx = list(df.index[missing_mask])

    if max_rows is not None:
        missing_idx = missing_idx[:max_rows]

    if not missing_idx:
        print("No missing evals to fill.")
        return df

    engine_path = find_stockfish_binary(stockfish_path)
    print(f"Using Stockfish at: {engine_path}")
    print(f"Rows queued for engine eval: {len(missing_idx)}")

    filled_df = df.copy()
    fen_cache = {}

    with chess.engine.SimpleEngine.popen_uci(engine_path) as engine:
        for idx in tqdm(missing_idx, desc="Computing missing evals", unit="row"):
            fen = filled_df.at[idx, "fen_after"]
            if fen not in fen_cache:
                fen_cache[fen] = score_fen_with_stockfish(
                    engine,
                    fen,
                    depth=depth,
                    time_limit=time_limit,
                )
            eval_cp, mate_in = fen_cache[fen]
            filled_df.at[idx, "eval_cp"] = eval_cp
            filled_df.at[idx, "mate_in"] = mate_in

    return filled_df


def load_partial_feb_2026(max_games: int = 10000) -> pd.DataFrame:
    text_stream = stream_text_from_zst_url(FEB_2026_URL)
    rows: List[Dict] = []

    with tqdm(total=max_games, desc="Parsing games", unit="game") as pbar:
        for _ in range(max_games):
            game = chess.pgn.read_game(text_stream)
            if game is None:
                break

            headers = game.headers
            board = game.board()

            result = headers.get("Result", "")
            white_elo = headers.get("WhiteElo")
            black_elo = headers.get("BlackElo")
            site = headers.get("Site", "")
            game_id = site.rstrip("/").split("/")[-1] if site else None

            node = game
            ply = 0

            while node.variations:
                next_node = node.variation(0)
                move = next_node.move
                ply += 1

                fen_before = board.fen()
                side_to_move = "white" if board.turn == chess.WHITE else "black"
                san = board.san(move)
                uci = move.uci()

                board.push(move)
                fen_after = board.fen()

                comment = next_node.comment or ""
                eval_cp, mate_in, clk_seconds = extract_eval_and_clock(comment)

                rows.append(
                    {
                        "game_id": game_id,
                        "date": headers.get("Date"),
                        "white_player": headers.get("White"),
                        "black_player": headers.get("Black"),
                        "white_elo": int(white_elo) if white_elo and white_elo.isdigit() else None,
                        "black_elo": int(black_elo) if black_elo and black_elo.isdigit() else None,
                        "result": result,
                        "label": result_to_label(result),
                        "time_control": headers.get("TimeControl"),
                        "eco": headers.get("ECO"),
                        "opening": headers.get("Opening"),
                        "ply": ply,
                        "side_to_move": side_to_move,
                        "san": san,
                        "uci": uci,
                        "fen_before": fen_before,
                        "fen_after": fen_after,
                        "eval_cp": eval_cp,
                        "mate_in": mate_in,
                        "clock_seconds_after_move": clk_seconds,
                    }
                )

                node = next_node

            pbar.update(1)

    return pd.DataFrame(rows)
        


In [19]:
# Step 1: Download and parse the dataset
df = load_partial_feb_2026(max_games=MAX_GAMES)

print(df.head())
print("Rows:", len(df))
print("Unique games:", df["game_id"].nunique())
        


Parsing games: 100%|███████████████████████████████████████████████████████████████████████████████| 10000/10000 [01:15<00:00, 132.58game/s]


    game_id        date    white_player  black_player  white_elo  black_elo  \
0  AJrdLVya  2026.02.01  Orangutanagram  bachess_2024       1696       1617   
1  AJrdLVya  2026.02.01  Orangutanagram  bachess_2024       1696       1617   
2  AJrdLVya  2026.02.01  Orangutanagram  bachess_2024       1696       1617   
3  AJrdLVya  2026.02.01  Orangutanagram  bachess_2024       1696       1617   
4  AJrdLVya  2026.02.01  Orangutanagram  bachess_2024       1696       1617   

  result      label time_control  eco  \
0    1-0  white_win         60+0  B12   
1    1-0  white_win         60+0  B12   
2    1-0  white_win         60+0  B12   
3    1-0  white_win         60+0  B12   
4    1-0  white_win         60+0  B12   

                                             opening  ply side_to_move san  \
0  Caro-Kann Defense: Advance Variation, Short Va...    1        white  e4   
1  Caro-Kann Defense: Advance Variation, Short Va...    2        black  c6   
2  Caro-Kann Defense: Advance Variation, Sho

In [20]:
# Step 2: Inspect evaluation coverage before engine fill
num_eval_cp = df["eval_cp"].notna().sum()
num_mate_in = df["mate_in"].notna().sum()
num_any_eval = (df["eval_cp"].notna() | df["mate_in"].notna()).sum()

print("Before Stockfish fill")
print("Total rows:", len(df))
print("Rows with eval_cp:", num_eval_cp)
print("Rows with mate_in:", num_mate_in)
print("Rows with any eval:", num_any_eval)
print("Fraction with any eval:", num_any_eval / len(df) if len(df) else 0)

df[df["eval_cp"].notna() | df["mate_in"].notna()].head(10)
        


Before Stockfish fill
Total rows: 667526
Rows with eval_cp: 67942
Rows with mate_in: 4642
Rows with any eval: 72584
Fraction with any eval: 0.1087358395028808


,game_id,date,white_player,black_player,white_elo,black_elo,result,label,time_control,eco,opening,ply,side_to_move,san,uci,fen_before,fen_after,eval_cp,mate_in,clock_seconds_after_move
1202,F2R0oAov,2026.02.01,xwenwenx,abhaykadekar1865,1952,1941,1-0,white_win,60+0,D00,Queen's Pawn Game,1,white,d4,d2d4,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,rnbqkbnr/pppppppp/8/8/3P4/8/PPP1PPPP/RNBQKBNR ...,17.0,NaN,60.0
1203,F2R0oAov,2026.02.01,xwenwenx,abhaykadekar1865,1952,1941,1-0,white_win,60+0,D00,Queen's Pawn Game,2,black,d5,d7d5,rnbqkbnr/pppppppp/8/8/3P4/8/PPP1PPPP/RNBQKBNR ...,rnbqkbnr/ppp1pppp/8/3p4/3P4/8/PPP1PPPP/RNBQKBN...,27.0,NaN,60.0
1204,F2R0oAov,2026.02.01,xwenwenx,abhaykadekar1865,1952,1941,1-0,white_win,60+0,D00,Queen's Pawn Game,3,white,e3,e2e3,rnbqkbnr/ppp1pppp/8/3p4/3P4/8/PPP1PPPP/RNBQKBN...,rnbqkbnr/ppp1pppp/8/3p4/3P4/4P3/PPP2PPP/RNBQKB...,3.0,NaN,60.0
1205,F2R0oAov,2026.02.01,xwenwenx,abhaykadekar1865,1952,1941,1-0,white_win,60+0,D00,Queen's Pawn Game,4,black,f5,f7f5,rnbqkbnr/ppp1pppp/8/3p4/3P4/4P3/PPP2PPP/RNBQKB...,rnbqkbnr/ppp1p1pp/8/3p1p2/3P4/4P3/PPP2PPP/RNBQ...,41.0,NaN,59.0
1206,F2R0oAov,2026.02.01,xwenwenx,abhaykadekar1865,1952,1941,1-0,white_win,60+0,D00,Queen's Pawn Game,5,white,c4,c2c4,rnbqkbnr/ppp1p1pp/8/3p1p2/3P4/4P3/PPP2PPP/RNBQ...,rnbqkbnr/ppp1p1pp/8/3p1p2/2PP4/4P3/PP3PPP/RNBQ...,34.0,NaN,60.0
1207,F2R0oAov,2026.02.01,xwenwenx,abhaykadekar1865,1952,1941,1-0,white_win,60+0,D00,Queen's Pawn Game,6,black,e6,e7e6,rnbqkbnr/ppp1p1pp/8/3p1p2/2PP4/4P3/PP3PPP/RNBQ...,rnbqkbnr/ppp3pp/4p3/3p1p2/2PP4/4P3/PP3PPP/RNBQ...,45.0,NaN,57.0
1208,F2R0oAov,2026.02.01,xwenwenx,abhaykadekar1865,1952,1941,1-0,white_win,60+0,D00,Queen's Pawn Game,7,white,Nf3,g1f3,rnbqkbnr/ppp3pp/4p3/3p1p2/2PP4/4P3/PP3PPP/RNBQ...,rnbqkbnr/ppp3pp/4p3/3p1p2/2PP4/4PN2/PP3PPP/RNB...,54.0,NaN,60.0
1209,F2R0oAov,2026.02.01,xwenwenx,abhaykadekar1865,1952,1941,1-0,white_win,60+0,D00,Queen's Pawn Game,8,black,Nf6,g8f6,rnbqkbnr/ppp3pp/4p3/3p1p2/2PP4/4PN2/PP3PPP/RNB...,rnbqkb1r/ppp3pp/4pn2/3p1p2/2PP4/4PN2/PP3PPP/RN...,47.0,NaN,57.0
1210,F2R0oAov,2026.02.01,xwenwenx,abhaykadekar1865,1952,1941,1-0,white_win,60+0,D00,Queen's Pawn Game,9,white,Nc3,b1c3,rnbqkb1r/ppp3pp/4pn2/3p1p2/2PP4/4PN2/PP3PPP/RN...,rnbqkb1r/ppp3pp/4pn2/3p1p2/2PP4/2N1PN2/PP3PPP/...,27.0,NaN,59.0
1211,F2R0oAov,2026.02.01,xwenwenx,abhaykadekar1865,1952,1941,1-0,white_win,60+0,D00,Queen's Pawn Game,10,black,c6,c7c6,rnbqkb1r/ppp3pp/4pn2/3p1p2/2PP4/2N1PN2/PP3PPP/...,rnbqkb1r/pp4pp/2p1pn2/3p1p2/2PP4/2N1PN2/PP3PPP...,30.0,NaN,56.0


In [21]:
# Step 3: Fill missing evaluation values with Stockfish
if FILL_MISSING_EVALS:
    df = fill_missing_evals_with_stockfish(
        df,
        stockfish_path=STOCKFISH_PATH,
        depth=ENGINE_DEPTH,
        time_limit=ENGINE_TIME_LIMIT,
        max_rows=ENGINE_MAX_ROWS,
    )
else:
    print("Skipping Stockfish fill.")
        


Using Stockfish at: /opt/homebrew/bin/stockfish
Rows queued for engine eval: 594942


Computing missing evals: 100%|███████████████████████████████████████████████████████████████████| 594942/594942 [08:00<00:00, 1237.98row/s]


In [25]:
# Step 4: Inspect the dataset after engine fill
num_eval_cp = df["eval_cp"].notna().sum()
num_mate_in = df["mate_in"].notna().sum()
num_any_eval = (df["eval_cp"].notna() | df["mate_in"].notna()).sum()

print("After Stockfish fill")
print("Total rows:", len(df))
print("Rows with eval_cp:", num_eval_cp)
print("Rows with mate_in:", num_mate_in)
print("Rows with any eval:", num_any_eval)
print("Fraction with any eval:", num_any_eval / len(df) if len(df) else 0)

df_eval = df[df["eval_cp"].notna() | df["mate_in"].notna()].copy()
print(df_eval.head(20).to_string(index=False))
        


After Stockfish fill
Total rows: 667526
Rows with eval_cp: 655405
Rows with mate_in: 12121
Rows with any eval: 667526
Fraction with any eval: 1.0
 game_id       date   white_player black_player  white_elo  black_elo result     label time_control eco                                               opening  ply side_to_move  san  uci                                                         fen_before                                                           fen_after  eval_cp  mate_in  clock_seconds_after_move
AJrdLVya 2026.02.01 Orangutanagram bachess_2024       1696       1617    1-0 white_win         60+0 B12 Caro-Kann Defense: Advance Variation, Short Variation    1        white   e4 e2e4           rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1          rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1     29.0      NaN                      60.0
AJrdLVya 2026.02.01 Orangutanagram bachess_2024       1696       1617    1-0 white_win         60+0 B12 Caro-Kann Defense: A

In [33]:
num_eval_cp = df["eval_cp"].notna().sum()
num_mate_in = df["mate_in"].notna().sum()
num_any_eval = (df["eval_cp"].notna() | df["mate_in"].notna()).sum()

print("Afer Stockfish fill")
print("Total rows:", len(df))
print("Rows with eval_cp:", num_eval_cp)
print("Rows with mate_in:", num_mate_in)
print("Rows with any eval:", num_any_eval)
print("Fraction with any eval:", num_any_eval / len(df) if len(df) else 0)

df[df["eval_cp"].notna() | df["mate_in"].notna()].head(10)

Afer Stockfish fill
Total rows: 667526
Rows with eval_cp: 655405
Rows with mate_in: 12121
Rows with any eval: 667526
Fraction with any eval: 1.0


,game_id,date,white_player,black_player,white_elo,black_elo,result,label,time_control,eco,opening,ply,side_to_move,san,uci,fen_before,fen_after,eval_cp,mate_in,clock_seconds_after_move
0,AJrdLVya,2026.02.01,Orangutanagram,bachess_2024,1696,1617,1-0,white_win,60+0,B12,"Caro-Kann Defense: Advance Variation, Short Va...",1,white,e4,e2e4,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR ...,29.0,NaN,60.0
1,AJrdLVya,2026.02.01,Orangutanagram,bachess_2024,1696,1617,1-0,white_win,60+0,B12,"Caro-Kann Defense: Advance Variation, Short Va...",2,black,c6,c7c6,rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR ...,rnbqkbnr/pp1ppppp/2p5/8/4P3/8/PPPP1PPP/RNBQKBN...,6.0,NaN,60.0
2,AJrdLVya,2026.02.01,Orangutanagram,bachess_2024,1696,1617,1-0,white_win,60+0,B12,"Caro-Kann Defense: Advance Variation, Short Va...",3,white,d4,d2d4,rnbqkbnr/pp1ppppp/2p5/8/4P3/8/PPPP1PPP/RNBQKBN...,rnbqkbnr/pp1ppppp/2p5/8/3PP3/8/PPP2PPP/RNBQKBN...,49.0,NaN,58.0
3,AJrdLVya,2026.02.01,Orangutanagram,bachess_2024,1696,1617,1-0,white_win,60+0,B12,"Caro-Kann Defense: Advance Variation, Short Va...",4,black,d5,d7d5,rnbqkbnr/pp1ppppp/2p5/8/3PP3/8/PPP2PPP/RNBQKBN...,rnbqkbnr/pp2pppp/2p5/3p4/3PP3/8/PPP2PPP/RNBQKB...,41.0,NaN,60.0
4,AJrdLVya,2026.02.01,Orangutanagram,bachess_2024,1696,1617,1-0,white_win,60+0,B12,"Caro-Kann Defense: Advance Variation, Short Va...",5,white,e5,e4e5,rnbqkbnr/pp2pppp/2p5/3p4/3PP3/8/PPP2PPP/RNBQKB...,rnbqkbnr/pp2pppp/2p5/3pP3/3P4/8/PPP2PPP/RNBQKB...,57.0,NaN,58.0
5,AJrdLVya,2026.02.01,Orangutanagram,bachess_2024,1696,1617,1-0,white_win,60+0,B12,"Caro-Kann Defense: Advance Variation, Short Va...",6,black,Bf5,c8f5,rnbqkbnr/pp2pppp/2p5/3pP3/3P4/8/PPP2PPP/RNBQKB...,rn1qkbnr/pp2pppp/2p5/3pPb2/3P4/8/PPP2PPP/RNBQK...,25.0,NaN,59.0
6,AJrdLVya,2026.02.01,Orangutanagram,bachess_2024,1696,1617,1-0,white_win,60+0,B12,"Caro-Kann Defense: Advance Variation, Short Va...",7,white,Nf3,g1f3,rn1qkbnr/pp2pppp/2p5/3pPb2/3P4/8/PPP2PPP/RNBQK...,rn1qkbnr/pp2pppp/2p5/3pPb2/3P4/5N2/PPP2PPP/RNB...,46.0,NaN,57.0
7,AJrdLVya,2026.02.01,Orangutanagram,bachess_2024,1696,1617,1-0,white_win,60+0,B12,"Caro-Kann Defense: Advance Variation, Short Va...",8,black,e6,e7e6,rn1qkbnr/pp2pppp/2p5/3pPb2/3P4/5N2/PPP2PPP/RNB...,rn1qkbnr/pp3ppp/2p1p3/3pPb2/3P4/5N2/PPP2PPP/RN...,7.0,NaN,58.0
8,AJrdLVya,2026.02.01,Orangutanagram,bachess_2024,1696,1617,1-0,white_win,60+0,B12,"Caro-Kann Defense: Advance Variation, Short Va...",9,white,Bd3,f1d3,rn1qkbnr/pp3ppp/2p1p3/3pPb2/3P4/5N2/PPP2PPP/RN...,rn1qkbnr/pp3ppp/2p1p3/3pPb2/3P4/3B1N2/PPP2PPP/...,36.0,NaN,56.0
9,AJrdLVya,2026.02.01,Orangutanagram,bachess_2024,1696,1617,1-0,white_win,60+0,B12,"Caro-Kann Defense: Advance Variation, Short Va...",10,black,Bxd3,f5d3,rn1qkbnr/pp3ppp/2p1p3/3pPb2/3P4/3B1N2/PPP2PPP/...,rn1qkbnr/pp3ppp/2p1p3/3pP3/3P4/3b1N2/PPP2PPP/R...,38.0,NaN,57.0


In [45]:

eval_series = df["eval_cp"]


min_eval = eval_series.min()
max_eval = eval_series.max()


num_pos = (eval_series > 0).sum()
num_neg = (eval_series < 0).sum()
num_zero = (eval_series == 0).sum()
total = len(eval_series)

print("=== eval_cp stats ===")
print("min:", min_eval)
print("max:", max_eval)

print("positive:", num_pos, "(", num_pos / total if total else 0, ")")
print("negative:", num_neg, "(", num_neg / total if total else 0, ")")
print("zero:", num_zero, "(", num_zero / total if total else 0, ")")

=== eval_cp stats ===
min: -8350.0
max: 8350.0
positive: 375484 ( 0.5625009362931181 )
negative: 275460 ( 0.41265808373007196 )
zero: 4461 ( 0.006682885760255031 )
